In [ ]:
def format_sse(data, event=None):
    """Format data for Server-Sent Events"""
    message = f"data: {json.dumps(data)}\n\n"
    if event:
        message = f"event: {event}\n{message}"
    return message

In [ ]:
# At the beginning of continue_preference function, add this check:
@app.route("/chat/preference:continue", methods=["POST"])
def continue_preference():
    try:
        json_data = request.get_json()
        uid = json_data.get('uid')
        user_input = json_data.get('user_input')
        history = json_data.get('history', [])
        
        # Check for streaming request header
        if request.headers.get('Accept') == 'text/event-stream':
            return continue_preference_stream()
            
    except:
        return jsonify({'error': 'Invalid JSON or missing uid/user_input/history'}), 400

In [ ]:
@app.route("/chat/preference:continue:stream", methods=["POST"])
def continue_preference_stream():
    """Streaming version of continue_preference for filter operations"""
    try:
        json_data = request.get_json()
        uid = json_data.get('uid')
        user_input = json_data.get('user_input')
        history = json_data.get('history', [])
    except:
        return jsonify({'error': 'Invalid JSON or missing uid/user_input/history'}), 400

    # Check if this is a filter request by analyzing the input
    filter_keywords = ['show recommendations', 'filter', 'only from', 'aged', 'age', 'profession', 'city', 'location']
    is_filter_request = any(keyword in user_input.lower() for keyword in filter_keywords)
    
    if is_filter_request:
        # For filter requests, return streaming response
        def generate():
            try:
                # Send initial message
                yield format_sse({
                    "type": "message",
                    "content": "I'll help you filter your recommendations. Let me process that for you..."
                }, "chat_message")
                
                # Start filtering process
                for chunk in live_filter_streaming(uid, user_input):
                    yield chunk
                    
            except Exception as e:
                yield format_sse({
                    "type": "error", 
                    "content": str(e)
                }, "chat_error")
        
        return Response(generate(), mimetype='text/event-stream')
    else:
        # For non-filter requests, redirect to normal chat endpoint
        return continue_preference()

In [ ]:
def format_sse(data, event=None):
    """Format data for Server-Sent Events"""
    message = f"data: {json.dumps(data)}\n\n"
    if event:
        message = f"event: {event}\n{message}"
    return message

def live_filter_streaming(uid: str, new_filter: str):
    """Streaming version of live_filter that yields progress updates"""
    def generate():
        try:
            # Send initial status
            yield format_sse({
                "status": "started",
                "message": "🔍 Filtering recommendations...",
                "progress": 10
            }, "filter_progress")
            
            # Fetch recommendations
            yield format_sse({
                "status": "fetching",
                "message": "📋 Fetching your current recommendations...",
                "progress": 30
            }, "filter_progress")
            
            recommendations = fetch_queue(uid, 'RECOMMENDATIONS')
            
            # Process with LLM
            yield format_sse({
                "status": "processing", 
                "message": "🤖 AI is analyzing your preferences...",
                "progress": 60
            }, "filter_progress")
            
            matches, filtered, matches_and_filtered = filter_cards(uid, recommendations, new_filter)
            
            # Update database
            yield format_sse({
                "status": "updating",
                "message": "💾 Updating your recommendations...", 
                "progress": 80
            }, "filter_progress")
            
            # Prepare results
            if len(matches) == 0:
                result = {
                    'RECOMMENDATIONS': recommendations, 
                    'RESPONSE': 'User filters do not satisfy any match, Please remove some filters.'
                }
            else:
                filtered_recommendations = []
                for card in recommendations:
                    if card['recommendation_uid'] in matches_and_filtered['MATCHED'].keys():
                        card['reason'] = matches_and_filtered['MATCHED'][card['recommendation_uid']][0]
                        filtered_recommendations.append(card)
                
                # Async update in background
                threading.Thread(
                    target=run_async_task, 
                    args=(update_matching_table_with_filter(uid, matches, filtered),)
                ).start()
                
                result = {
                    'RECOMMENDATIONS': filtered_recommendations, 
                    'RESPONSE': 'Recommendations have been updated as per your request.'
                }
            
            # Send final results
            yield format_sse({
                "status": "completed",
                "message": f"✅ Found {len(result['RECOMMENDATIONS'])} matching recommendations!",
                "progress": 100,
                "data": result
            }, "filter_complete")
            
        except Exception as e:
            log.error(f"Error in live_filter_streaming: {e}")
            yield format_sse({
                "status": "error",
                "message": f"❌ Error occurred: {str(e)}",
                "progress": 0
            }, "filter_error")
    
    return generate()